In [1]:
import pandas as pd
from pathlib import Path
import sys

def processa_log_tesi(base_repo_path):
    """
    Esegue i seguenti passaggi:
    1. Trova tutte le cartelle che iniziano per 'S' nel percorso base.
    2. In ognuna di esse, cerca ricorsivamente 'predictions_log.csv'.
    3. ESTRAE 'myo_inf_time' e CALCOLA 'processing_time_ms' (diff da 'timestamp').
    4. Concatena i dati in DataFrame separati.
    5. Calcola le statistiche per entrambi.
    6. Salva i dati aggregati e le statistiche in cartelle separate:
       - ../data/all/myo_inf/
       - ../data/all/proc_time/
    """
    base_path = Path(base_repo_path)
    
    # 1. Definizione dei percorsi
    script_dir = Path.cwd().resolve()
    
    # Definisce il percorso di OUTPUT per MYO_INF_TIME
    myo_inf_output_dir = script_dir.parent / "data" / "all" / "myo_inf"
    
    # NUOVO: Definisce il percorso di OUTPUT per PROCESSING_TIME
    proc_time_output_dir = script_dir.parent / "data" / "all" / "proc_time"
    
    # Crea le cartelle di output se non esistono
    try:
        myo_inf_output_dir.mkdir(parents=True, exist_ok=True)
        proc_time_output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Cartella output 'myo_inf' creata (o già esistente): {myo_inf_output_dir}")
        print(f"Cartella output 'proc_time' creata (o già esistente): {proc_time_output_dir}")
    except Exception as e:
        print(f"Errore fatale: Impossibile creare le cartelle di output.")
        print(f"Dettagli: {e}")
        sys.exit(1) # Esce dallo script

    # File di output per MYO_INF_TIME
    myo_inf_csv_file = myo_inf_output_dir / "compiled_myo_inf_time.csv"
    myo_inf_stats_file = myo_inf_output_dir / "statistics_myo_inf.txt"

    # NUOVO: File di output per PROCESSING_TIME
    proc_time_csv_file = proc_time_output_dir / "compiled_processing_time_ms.csv"
    proc_time_stats_file = proc_time_output_dir / "statistics_proc_time.txt"

    # Liste per contenere tutte le serie di dati
    all_myo_data_series = []
    all_proc_time_series = [] # NUOVO
    
    print(f"\nInizio ricerca in: {base_path}")

    # 2. Trova tutte le cartelle che iniziano per 'S'
    s_folders = [f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')]
    
    if not s_folders:
        print("Attenzione: Nessuna cartella che inizia con 'S' trovata.")
        return

    print(f"Trovate {len(s_folders)} cartelle 'S': {[f.name for f in s_folders]}")

    # 3. Cerca i file CSV ed estrai i dati
    file_trovati_myo = 0
    file_trovati_proc = 0 # NUOVO

    for folder in s_folders:
        for csv_file in folder.rglob("predictions_log.csv"):
            try:
                # Leggi il file CSV
                df = pd.read_csv(csv_file)
                
                # --- A. Processa 'myo_inf_time' (Logica esistente) ---
                if "myo_inf_time" in df.columns:
                    all_myo_data_series.append(df["myo_inf_time"])
                    file_trovati_myo += 1
                else:
                    print(f"  > Attenzione: Colonna 'myo_inf_time' non trovata in {csv_file}")
                
                # --- B. NUOVO: Processa 'timestamp' per processing time ---
                if "timestamp" in df.columns:
                    if len(df) < 2:
                        print(f"  > Info: File {csv_file} ha meno di 2 righe, impossibile calcolare diff per 'timestamp'.")
                    else:
                        try:
                            # Converte la colonna 'timestamp' in oggetti datetime
                            # Formato atteso: HH_MM_SS_MS (es. 11_37_55_290)
                            timestamps = pd.to_datetime(df["timestamp"], format="%H_%M_%S_%f", errors='coerce')
                            
                            # Calcola la differenza tra righe consecutive
                            proc_times = timestamps.diff()
                            
                            # Rimuovi il primo valore (che è NaT/NaN)
                            proc_times = proc_times.dropna()
                            
                            if not proc_times.empty:
                                # Converte i Timedelta in millisecondi
                                proc_times_ms = proc_times.dt.total_seconds() * 1000
                                all_proc_time_series.append(proc_times_ms)
                                file_trovati_proc += 1
                            else:
                                print(f"  > Attenzione: Dati 'timestamp' non validi o insufficienti in {csv_file}")
                                
                        except Exception as e:
                            print(f"  > ERRORE durante conversione 'timestamp' in {csv_file}: {e}")
                else:
                    print(f"  > Attenzione: Colonna 'timestamp' non trovata in {csv_file}")
            
            except pd.errors.EmptyDataError:
                print(f"  > Attenzione: File vuoto ignorato {csv_file}")
            except Exception as e:
                print(f"  > Errore durante la lettura di {csv_file}: {e}")

    # --- 4. Aggrega e Salva 'myo_inf_time' ---
    if not all_myo_data_series:
        print("\nOperazione terminata: Nessun dato 'myo_inf_time' è stato trovato.")
    else:
        print(f"\nTrovati e processati {file_trovati_myo} file per 'myo_inf_time'.")
        
        combined_myo_data = pd.concat(all_myo_data_series, ignore_index=True)
        combined_myo_data.name = "myo_inf_time"

        try:
            combined_myo_data.to_csv(myo_inf_csv_file, index=False, header=True)
            print(f"Dati 'myo_inf_time' aggregati salvati con successo in:\n{myo_inf_csv_file}")
        except Exception as e:
            print(f"\nErrore durante il salvataggio del file CSV 'myo_inf_time': {e}")
            return

        # Calcola e salva statistiche per 'myo_inf_time'
        print("Calcolo statistiche 'myo_inf_time'...")
        try:
            stats_myo = combined_myo_data.describe()
            
            stats_content = f"""
Statistiche per 'myo_inf_time'
-----------------------------------
Cartella di origine: {base_path}
File processati:     {file_trovati_myo}
-----------------------------------

Metriche principali:
  Media:           {stats_myo.loc['mean']:.6f}
  Deviazione Std:  {stats_myo.loc['std']:.6f}

Metriche descrittive:
  Campioni totali: {int(stats_myo.loc['count'])}
  Valore Minimo:   {stats_myo.loc['min']:.6f}
  25° Percentile:  {stats_myo.loc['25%']:.6f}
  Mediana (50°):   {stats_myo.loc['50%']:.6f}
  75° Percentile:  {stats_myo.loc['75%']:.6f}
  Valore Massimo:  {stats_myo.loc['max']:.6f}
"""
            with open(myo_inf_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            
            print(f"Statistiche 'myo_inf_time' salvate con successo in:\n{myo_inf_stats_file}")

        except Exception as e:
            print(f"Errore durante il calcolo o salvataggio delle statistiche 'myo_inf_time': {e}")

    # --- 5. NUOVO: Aggrega e Salva 'processing_time_ms' ---
    if not all_proc_time_series:
        print("\nOperazione terminata: Nessun dato 'processing_time' è stato calcolato.")
    else:
        print(f"\nTrovati e processati {file_trovati_proc} file per 'processing_time'.")
        
        combined_proc_data = pd.concat(all_proc_time_series, ignore_index=True)
        combined_proc_data.name = "processing_time_ms"

        try:
            combined_proc_data.to_csv(proc_time_csv_file, index=False, header=True)
            print(f"Dati 'processing_time_ms' aggregati salvati con successo in:\n{proc_time_csv_file}")
        except Exception as e:
            print(f"\nErrore during salvataggio del file CSV 'processing_time_ms': {e}")
            return

        # Calcola e salva statistiche per 'processing_time_ms'
        print("Calcolo statistiche 'processing_time_ms'...")
        try:
            stats_proc = combined_proc_data.describe()
            
            stats_content = f"""
Statistiche per 'processing_time_ms' (Tempo tra timestamp consecutivi)
-----------------------------------
Cartella di origine: {base_path}
File processati:     {file_trovati_proc}
-----------------------------------

Metriche principali (in millisecondi):
  Media:           {stats_proc.loc['mean']:.6f} ms
  Deviazione Std:  {stats_proc.loc['std']:.6f} ms

Metriche descrittive (in millisecondi):
  Campioni totali: {int(stats_proc.loc['count'])}
  Valore Minimo:   {stats_proc.loc['min']:.6f} ms
  25° Percentile:  {stats_proc.loc['25%']:.6f} ms
  Mediana (50°):   {stats_proc.loc['50%']:.6f} ms
  75° Percentile:  {stats_proc.loc['75%']:.6f} ms
  Valore Massimo:  {stats_proc.loc['max']:.6f} ms
"""
            with open(proc_time_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            
            print(f"Statistiche 'processing_time_ms' salvate con successo in:\n{proc_time_stats_file}")

        except Exception as e:
            print(f"Errore durante il calcolo o salvataggio delle statistiche 'processing_time_ms': {e}")

    print("\n--- Operazione completata ---")


# --- INIZIO SCRIPT ---
if __name__ == "__main__":
    # Inserisci qui il tuo percorso base. 
    # L'uso di r"..." (raw string) previene problemi con i backslash \ di Windows.
    percorso_base = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    processa_log_tesi(percorso_base)

Cartella output 'myo_inf' creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf
Cartella output 'proc_time' creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\proc_time

Inizio ricerca in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Trovate 12 cartelle 'S': ['S03', 'S08', 'S10', 'S12', 'S13', 'S15', 'S16', 'S17', 'S18', 'S19', 'S20', 'S21']

Trovati e processati 63 file per 'myo_inf_time'.
Dati 'myo_inf_time' aggregati salvati con successo in:
C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf\compiled_myo_inf_time.csv
Calcolo statistiche 'myo_inf_time'...
Statistiche 'myo_inf_time' salvate con successo in:
C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf\statistics_myo_inf.txt

Trovati e processati 63 file per 'processing_time'.
Dati 'processing_time_ms' aggregati salvati con successo in:
C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\proc_time\compiled_processing_time_ms.csv
Calcolo stati

In [5]:
import pandas as pd
from pathlib import Path
import sys

def processa_realtime_log(base_repo_path):
    """
    Esegue i seguenti passaggi:
    1. Cerca 'realtime_log.csv' in tutte le sottocartelle 'S' del percorso base.
    2. Controlla la colonna 'gate_cv'.
    3. Se gate_cv == 1:
       - Estrae 'cv_inf_ms' e 'proc_time_ms'.
       - Salva i dati aggregati in 'gate_1_data.csv'.
       - Salva le statistiche di entrambe le colonne in 'gate_1_statistics.txt'.
    4. Se gate_cv == 0:
       - Estrae 'proc_time_ms'.
       - Salva i dati aggregati in 'gate_0_proc_time.csv'.
       - Salva le statistiche di questa colonna in 'gate_0_statistics.txt'.
    5. Salva tutto in ../data/all/cv_inf (relativo allo script).
    """
    
    # 1. Definizione dei percorsi
    
    # Percorso INPUT: Dove si trovano i dati
    base_path = Path(base_repo_path)
    
    # Percorso OUTPUT: Relativo alla posizione dello script
    try:
        script_dir = Path(__file__).resolve().parent
        output_dir = script_dir.parent / "data" / "all" / "cv_inf"
    except NameError:
        print("--- ERRORE ---")
        print("Sembra che tu stia eseguendo questo codice in un ambiente interattivo (es. Jupyter).")
        print("In questo caso, '__file__' non è definito.")
        print("Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.")
        print("Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.")
        script_dir = Path.cwd().resolve()
        output_dir = script_dir.parent / "data" / "all" / "cv_inf"
        print("----------------")

    # Crea la cartella di output
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Cartella di output creata (o già esistente): {output_dir}")
    except Exception as e:
        print(f"Errore fatale: Impossibile creare la cartella di output {output_dir}.")
        print(f"Dettagli: {e}")
        sys.exit(1)

    # File di output
    gate_1_csv_file = output_dir / "gate_1_data.csv"
    gate_1_stats_file = output_dir / "gate_1_statistics.txt"
    gate_0_csv_file = output_dir / "gate_0_proc_time.csv"
    gate_0_stats_file = output_dir / "gate_0_statistics.txt"

    # Liste per contenere i dati estratti
    gate_1_data_list = []
    gate_0_data_list = []
    
    # Colonne richieste
    colonne_richieste = ['cv_inf_ms', 'proc_time_ms', 'gate_cv']

    print(f"Inizio ricerca dati in: {base_path}")

    # 2. Trova tutte le cartelle che iniziano per 'S'
    s_folders = [f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')]
    
    if not s_folders:
        print("Attenzione: Nessuna cartella che inizia con 'S' trovata.")
        return

    print(f"Trovate {len(s_folders)} cartelle 'S'. Inizio scansione...")

    # 3. Cerca i file CSV e processali
    file_trovati = 0
    for folder in s_folders:
        for csv_file in folder.rglob("realtime_log.csv"):
            try:
                df = pd.read_csv(csv_file)
                file_trovati += 1

                # Controlla se le colonne necessarie esistono
                if not all(col in df.columns for col in colonne_richieste):
                    print(f"  > Attenzione: File ignorato {csv_file}")
                    print(f"    Mancano una o più colonne: {colonne_richieste}")
                    continue

                # --- Caso 1: gate_cv == 1 ---
                df_gate_1 = df.loc[df['gate_cv'] == 1, ['cv_inf_ms', 'proc_time_ms']]
                if not df_gate_1.empty:
                    gate_1_data_list.append(df_gate_1)

                # --- Caso 2: gate_cv == 0 ---
                df_gate_0 = df.loc[df['gate_cv'] == 0, ['proc_time_ms']]
                if not df_gate_0.empty:
                    gate_0_data_list.append(df_gate_0)

            except pd.errors.EmptyDataError:
                print(f"  > Attenzione: File vuoto ignorato {csv_file}")
            except Exception as e:
                print(f"  > Errore durante la lettura di {csv_file}: {e}")

    print(f"\nScansione completata. Processati {file_trovati} file 'realtime_log.csv'.")

    # 4. Processa e salva i dati per GATE_CV == 1
    if gate_1_data_list:
        print("\n--- Processando dati per gate_cv == 1 ---")
        combined_gate_1 = pd.concat(gate_1_data_list, ignore_index=True)
        
        # Salva CSV
        try:
            combined_gate_1.to_csv(gate_1_csv_file, index=False)
            print(f"Dati 'gate_1' salvati con successo in:\n{gate_1_csv_file}")
        except Exception as e:
            print(f"Errore durante il salvataggio di {gate_1_csv_file}: {e}")

        # Calcola e salva statistiche
        try:
            stats_content = f"""
Statistiche per gate_cv == 1
-----------------------------------
File 'realtime_log.csv' processati: {file_trovati}
Campioni totali (con gate_cv == 1): {len(combined_gate_1)}
-----------------------------------

Colonna: 'cv_inf_ms'
  Media:           {combined_gate_1['cv_inf_ms'].mean():.6f}
  Deviazione Std:  {combined_gate_1['cv_inf_ms'].std():.6f}
  Minimo:          {combined_gate_1['cv_inf_ms'].min():.6f}
  Mediana:         {combined_gate_1['cv_inf_ms'].median():.6f}
  Massimo:         {combined_gate_1['cv_inf_ms'].max():.6f}

Colonna: 'proc_time_ms'
  Media:           {combined_gate_1['proc_time_ms'].mean():.6f}
  Deviazione Std:  {combined_gate_1['proc_time_ms'].std():.6f}
  Minimo:          {combined_gate_1['proc_time_ms'].min():.6f}
  Mediana:         {combined_gate_1['proc_time_ms'].median():.6f}
  Massimo:         {combined_gate_1['proc_time_ms'].max():.6f}
"""
            with open(gate_1_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            print(f"Statistiche 'gate_1' salvate con successo in:\n{gate_1_stats_file}")

        except Exception as e:
            print(f"Errore durante il calcolo/salvataggio delle statistiche 'gate_1': {e}")
    else:
        print("\nNessun dato trovato per gate_cv == 1.")

    # 5. Processa e salva i dati per GATE_CV == 0
    if gate_0_data_list:
        print("\n--- Processando dati per gate_cv == 0 ---")
        combined_gate_0 = pd.concat(gate_0_data_list, ignore_index=True)
        
        # Salva CSV
        try:
            combined_gate_0.to_csv(gate_0_csv_file, index=False)
            print(f"Dati 'gate_0' salvati con successo in:\n{gate_0_csv_file}")
        except Exception as e:
            print(f"Errore durante il salvataggio di {gate_0_csv_file}: {e}")

        # Calcola e salva statistiche
        try:
            stats_content = f"""
Statistiche per gate_cv == 0
-----------------------------------
File 'realtime_log.csv' processati: {file_trovati}
Campioni totali (con gate_cv == 0): {len(combined_gate_0)}
-----------------------------------

Colonna: 'proc_time_ms'
  Media:           {combined_gate_0['proc_time_ms'].mean():.6f}
  Deviazione Std:  {combined_gate_0['proc_time_ms'].std():.6f}
  Minimo:          {combined_gate_0['proc_time_ms'].min():.6f}
  Mediana:         {combined_gate_0['proc_time_ms'].median():.6f}
  Massimo:         {combined_gate_0['proc_time_ms'].max():.6f}
"""
            with open(gate_0_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            print(f"Statistiche 'gate_0' salvate con successo in:\n{gate_0_stats_file}")

        except Exception as e:
            print(f"Errore duringo il calcolo/salvataggio delle statistiche 'gate_0': {e}")
    else:
        print("\nNessun dato trovato per gate_cv == 0.")

    print("\n--- Operazione completata ---")

# --- INIZIO SCRIPT ---
if __name__ == "__main__":
    # Percorso INPUT: dove si trovano i dati da analizzare.
    percorso_base_dati = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    processa_realtime_log(percorso_base_dati)

--- ERRORE ---
Sembra che tu stia eseguendo questo codice in un ambiente interattivo (es. Jupyter).
In questo caso, '__file__' non è definito.
Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.
Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.
----------------
Cartella di output creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\cv_inf
Inizio ricerca dati in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Trovate 12 cartelle 'S'. Inizio scansione...

Scansione completata. Processati 49 file 'realtime_log.csv'.

--- Processando dati per gate_cv == 1 ---
Dati 'gate_1' salvati con successo in:
C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\cv_inf\gate_1_data.csv
Statistiche 'gate_1' salvate con successo in:
C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\cv_inf\gate_1_statistics.txt

--- Processando dati per gate_cv == 0 ---
Dati 'gate_0' salvati con successo in:
C:\Users\nicol\Thesis\pyl_est\offli

In [8]:
import pandas as pd
from pathlib import Path
import sys

def processa_classes_log(base_repo_path):
    """
    Esegue i seguenti passaggi:
    1. Cerca 'classes_log.csv' in tutte le sottocartelle 'S' del percorso base.
    2. Controlla la colonna 'label'.
    3. Se label != 0:
       - Estrae la colonna 'transition_time'.
    4. Salva i dati aggregati in 'classes_log_transition_time.csv'.
    5. Salva le statistiche in 'classes_log_statistics.txt',
       includendo un conteggio dei campioni per ciascun soggetto 'S'.
    6. Salva tutto in ../data/all/myo_inf (relativo allo script).
    """
    
    # 1. Definizione dei percorsi
    
    # Percorso INPUT: Dove si trovano i dati
    base_path = Path(base_repo_path)
    
    # Percorso OUTPUT: Relativo alla posizione dello script
    try:
        script_dir = Path(__file__).resolve().parent
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
    except NameError:
        print("--- ERRORE RILEVATO (Jupyter/Interattivo) ---")
        print("Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.")
        print("Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.")
        script_dir = Path.cwd().resolve()
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
        print("---------------------------------------------")

    # Crea la cartella di output
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Cartella di output creata (o già esistente): {output_dir}")
    except Exception as e:
        print(f"Errore fatale: Impossibile creare la cartella di output {output_dir}.")
        print(f"Dettagli: {e}")
        sys.exit(1)

    # File di output (con nomi specifici per evitare sovrascritture)
    output_csv_file = output_dir / "classes_log_transition_time.csv"
    output_stats_file = output_dir / "classes_log_statistics.txt"

    # Lista per contenere i dati estratti
    data_list = []
    
    # --- NOVITÀ ---
    # Dizionario per tenere traccia dei conteggi per soggetto
    conteggi_per_soggetto = {}
    
    # Colonne richieste
    colonne_richieste = ['label', 'transition_time']

    print(f"Inizio ricerca dati in: {base_path}")

    # 2. Trova tutte le cartelle che iniziano per 'S'
    s_folders = [f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')]
    
    if not s_folders:
        print("Attenzione: Nessuna cartella che inizia con 'S' trovata.")
        return

    print(f"Trovate {len(s_folders)} cartelle 'S'. Inizio scansione...")

    # 3. Cerca i file CSV e processali
    file_trovati = 0
    for folder in s_folders:
        
        # --- NOVITÀ ---
        # Contatore per il soggetto corrente
        conteggio_soggetto_corrente = 0
        
        for csv_file in folder.rglob("classes_log.csv"):
            try:
                df = pd.read_csv(csv_file)
                file_trovati += 1

                if not all(col in df.columns for col in colonne_richieste):
                    print(f"  > Attenzione: File ignorato {csv_file}")
                    print(f"    Mancano una o più colonne: {colonne_richieste}")
                    continue

                # --- Applica filtro: label != 0 ---
                df_filtered = df.loc[df['label'] != 0, ['transition_time']]
                
                if not df_filtered.empty:
                    data_list.append(df_filtered)
                    
                    # --- NOVITÀ ---
                    # Aggiorna il conteggio per questo soggetto
                    conteggio_soggetto_corrente += len(df_filtered)

            except pd.errors.EmptyDataError:
                print(f"  > Attenzione: File vuoto ignorato {csv_file}")
            except Exception as e:
                print(f"  > Errore durante la lettura di {csv_file}: {e}")
        
        # --- NOVITÀ ---
        # Salva il conteggio totale per il soggetto, anche se è 0
        conteggi_per_soggetto[folder.name] = conteggio_soggetto_corrente
        if conteggio_soggetto_corrente > 0:
            print(f"  > Trovati {conteggio_soggetto_corrente} campioni per il soggetto {folder.name}")

    print(f"\nScansione completata. Processati {file_trovati} file 'classes_log.csv'.")

    # 4. Processa e salva i dati
    if data_list:
        print("\n--- Processando dati per label != 0 ---")
        combined_data = pd.concat(data_list, ignore_index=True)
        
        # Salva CSV
        try:
            combined_data.to_csv(output_csv_file, index=False)
            print(f"Dati 'transition_time' (label != 0) salvati con successo in:\n{output_csv_file}")
        except Exception as e:
            print(f"Errore durante il salvataggio di {output_csv_file}: {e}")

        # Calcola e salva statistiche
        try:
            stats_data = combined_data['transition_time']
            
            # --- NOVITÀ ---
            # Costruisci la stringa per il riepilogo per soggetto
            stats_soggetti_lines = ["\nConteggio campioni per soggetto (dove label != 0):"]
            if not conteggi_per_soggetto:
                stats_soggetti_lines.append("  Nessun soggetto analizzato.")
            else:
                # Ordina i soggetti per nome (es. S01, S02, S10)
                for soggetto, conteggio in sorted(conteggi_per_soggetto.items()):
                    stats_soggetti_lines.append(f"  - {soggetto}: {conteggio} campioni")
            
            stats_soggetti = "\n".join(stats_soggetti_lines)

            # --- MODIFICATO ---
            # Aggiunto il blocco 'stats_soggetti' al file di testo
            stats_content = f"""
Statistiche per 'transition_time' (dove label != 0)
---------------------------------------------------
File 'classes_log.csv' processati: {file_trovati}
Campioni totali (con label != 0):  {len(stats_data)}
---------------------------------------------------
{stats_soggetti}
---------------------------------------------------

Metriche aggregate ('transition_time'):
  Media:           {stats_data.mean():.6f}
  Deviazione Std:  {stats_data.std():.6f}
  Minimo:          {stats_data.min():.6f}
  25° Percentile:  {stats_data.quantile(0.25):.6f}
  Mediana (50°):   {stats_data.median():.6f}
  75° Percentile:  {stats_data.quantile(0.75):.6f}
  Massimo:         {stats_data.max():.6f}
"""
            with open(output_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            print(f"Statistiche 'transition_time' (con conteggi soggetto) salvate con successo in:\n{output_stats_file}")

        except Exception as e:
            print(f"Errore durante il calcolo/salvataggio delle statistiche: {e}")
    else:
        print("\nNessun dato trovato con 'label' diverso da 0.")

    print("\n--- Operazione completata ---")

# --- INIZIO SCRIPT ---
if __name__ == "__main__":
    # Percorso INPUT: dove si trovano i dati da analizzare.
    percorso_base_dati = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    processa_classes_log(percorso_base_dati)

--- ERRORE RILEVATO (Jupyter/Interattivo) ---
Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.
Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.
---------------------------------------------
Cartella di output creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf
Inizio ricerca dati in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Trovate 12 cartelle 'S'. Inizio scansione...
  > Trovati 35 campioni per il soggetto S03
  > Trovati 39 campioni per il soggetto S08
  > Trovati 33 campioni per il soggetto S10
  > Attenzione: File vuoto ignorato C:\Users\nicol\Thesis\DATA real time\_test_raw\S12\test2b\5\classes_log.csv
  > Trovati 36 campioni per il soggetto S12
  > Trovati 36 campioni per il soggetto S13
  > Trovati 37 campioni per il soggetto S15
  > Trovati 30 campioni per il soggetto S16
  > Trovati 36 campioni per il soggetto S17
  > Trovati 31 campioni per il soggetto S18
  > Trovati 31 campioni pe

In [11]:
import pandas as pd
from pathlib import Path
import sys

def processa_emg_log(base_repo_path):
    """
    Esegue i seguenti passaggi:
    1. Cerca 'raw_emg_imu.csv' in tutte le sottocartelle 'S'.
    2. Per ogni file, calcola una frequenza di campionamento stimata.
    3. Logica Frequenza:
       - N_valid = (Righe Totali) - (Conteggio del primo timestamp)
       - Delta_T = (Ultimo timestamp) - (Primo timestamp) [in secondi]
       - Frequenza (Hz) = N_valid / Delta_T
    4. Salva tutte le frequenze in 'frequencies_emg.csv'.
    5. Salva le statistiche in 'frequencies_emg_statistics.txt'.
    6. Salva tutto in ../data/all/myo_inf (relativo allo script).
    """
    
    # 1. Definizione dei percorsi
    
    # Percorso INPUT: Dove si trovano i dati
    base_path = Path(base_repo_path)
    
    # Percorso OUTPUT: Relativo alla posizione dello script
    try:
        script_dir = Path(__file__).resolve().parent
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
    except NameError:
        print("--- ERRORE RILEVATO (Jupyter/Interattivo) ---")
        print("Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.")
        print("Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.")
        script_dir = Path.cwd().resolve()
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
        print("---------------------------------------------")

    # Crea la cartella di output
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Cartella di output creata (o già esistente): {output_dir}")
    except Exception as e:
        print(f"Errore fatale: Impossibile creare la cartella di output {output_dir}.")
        print(f"Dettagli: {e}")
        sys.exit(1)

    # File di output
    output_csv_file = output_dir / "frequencies_emg.csv"
    output_stats_file = output_dir / "frequencies_emg_statistics.txt"

    # Lista per contenere tutte le frequenze calcolate
    all_frequencies = []
    
    print(f"Inizio ricerca dati in: {base_path}")

    # 2. Trova tutte le cartelle che iniziano per 'S'
    s_folders = [f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')]
    
    if not s_folders:
        print("Attenzione: Nessuna cartella che inizia con 'S' trovata.")
        return

    print(f"Trovate {len(s_folders)} cartelle 'S'. Inizio scansione...")

    # 3. Cerca i file CSV e processali
    file_trovati = 0
    for folder in s_folders:
        for csv_file in folder.rglob("raw_emg_imu.csv"):
            try:
                # Leggi solo la colonna 'timestamps' come stringa
                df = pd.read_csv(
                    csv_file, 
                    usecols=['Timestamps'], 
                    dtype={'Timestamps': str},
                    skipinitialspace=True # Rimuove spazi bianchi
                )
                
                # Rimuovi eventuali righe 'None' o 'NaN'
                df.dropna(inplace=True)
                
                total_rows = len(df)

                # Servono almeno 2 righe per calcolare una durata
                if total_rows < 2:
                    print(f"  > Attenzione: File ignorato {csv_file} (meno di 2 righe)")
                    continue

                # ----------------------------------------------------
                # Inizio logica di calcolo
                # ----------------------------------------------------
                
                # 1. Trova primo e ultimo timestamp (come stringhe)
                first_ts_value = df['Timestamps'].iloc[0]
                last_ts_value = df['Timestamps'].iloc[-1]
                
                # 2. Conta le occorrenze del primo timestamp
                first_ts_count = (df['Timestamps'] == first_ts_value).sum()
                
                # 3. Calcola N_valid (come da tua richiesta)
                n_valid = total_rows - first_ts_count
                
                if n_valid <= 0:
                    print(f"  > Attenzione: File ignorato {csv_file} (N_valid = {n_valid})")
                    continue

                # 4. Calcola Delta_T (Durata)
                # Converti le stringhe in oggetti datetime
                # Il formato %H_%M_%S_%f gestisce HH_MM_SS_mmm (es. 043 -> 43000 microsecondi)
                start_time = pd.to_datetime(first_ts_value, format='%H_%M_%S_%f')
                end_time = pd.to_datetime(last_ts_value, format='%H_%M_%S_%f')
                
                # Gestione del caso "superamento mezzanotte" (es. 23:59 -> 00:01)
                if end_time < start_time:
                    end_time += pd.Timedelta(days=1)
                
                # Calcola la durata in secondi
                delta_t_seconds = (end_time - start_time).total_seconds()
                
                if delta_t_seconds == 0:
                    print(f"  > Attenzione: File ignorato {csv_file} (Durata totale è 0.0s)")
                    continue

                # 5. Calcola Frequenza (Hz)
                # Formula: Campioni / Secondi
                frequency_hz = n_valid / delta_t_seconds
                # ----------------------------------------------------
                
                all_frequencies.append(frequency_hz)
                file_trovati += 1
                print(f"  > Processato {folder.name}/{csv_file.name}: {frequency_hz:.2f} Hz")

            except pd.errors.EmptyDataError:
                print(f"  > Attenzione: File vuoto ignorato {csv_file}")
            except KeyError:
                print(f"  > Attenzione: Colonna 'timestamps' non trovata in {csv_file}")
            except ValueError as e:
                print(f"  > Errore parsing timestamp in {csv_file}: {e}")
            except Exception as e:
                print(f"  > Errore sconosciuto durante lettura di {csv_file}: {e}")

    print(f"\nScansione completata. Processati {file_trovati} file 'raw_emg_imu.csv'.")

    # 4. Processa e salva i dati
    if all_frequencies:
        print("\n--- Processando frequenze calcolate ---")
        freq_df = pd.DataFrame(all_frequencies, columns=['calculated_frequency_hz'])
        
        # Salva CSV
        try:
            freq_df.to_csv(output_csv_file, index=False)
            print(f"Dati 'frequencies_emg.csv' salvati con successo in:\n{output_csv_file}")
        except Exception as e:
            print(f"Errore duringo il salvataggio di {output_csv_file}: {e}")

        # Calcola e salva statistiche
        try:
            stats_content = f"""
Statistiche per la Frequenza di Campionamento EMG (Hz)
---------------------------------------------------
File 'raw_emg_imu.csv' processati: {file_trovati}

Nota sulla metodologia (come da richiesta):
- N_valid = (Righe Totali) - (Conteggio del primo timestamp)
- Delta_T = (Ultimo timestamp) - (Primo timestamp) [secondi]
- Frequenza = N_valid / Delta_T
---------------------------------------------------

Metriche aggregate:
  Media:           {freq_df['calculated_frequency_hz'].mean():.3f} Hz
  Deviazione Std:  {freq_df['calculated_frequency_hz'].std():.3f} Hz
  Minimo:          {freq_df['calculated_frequency_hz'].min():.3f} Hz
  25° Percentile:  {freq_df['calculated_frequency_hz'].quantile(0.25):.3f} Hz
  Mediana (50°):   {freq_df['calculated_frequency_hz'].median():.3f} Hz
  75° Percentile:  {freq_df['calculated_frequency_hz'].quantile(0.75):.3f} Hz
  Massimo:         {freq_df['calculated_frequency_hz'].max():.3f} Hz
"""
            with open(output_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            print(f"Statistiche 'frequencies_emg_statistics.txt' salvate con successo in:\n{output_stats_file}")

        except Exception as e:
            print(f"Errore durante il calcolo/salvataggio delle statistiche: {e}")
    else:
        print("\nNessuna frequenza è stata calcolata.")

    print("\n--- Operazione completata ---")

# --- INIZIO SCRIPT ---
if __name__ == "__main__":
    # Percorso INPUT: dove si trovano i dati da analizzare.
    percorso_base_dati = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    processa_emg_log(percorso_base_dati)

--- ERRORE RILEVATO (Jupyter/Interattivo) ---
Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.
Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.
---------------------------------------------
Cartella di output creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf
Inizio ricerca dati in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Trovate 12 cartelle 'S'. Inizio scansione...
  > Processato S03/raw_emg_imu.csv: 165.44 Hz
  > Processato S03/raw_emg_imu.csv: 121.84 Hz
  > Processato S03/raw_emg_imu.csv: 123.05 Hz
  > Processato S03/raw_emg_imu.csv: 95.63 Hz
  > Processato S03/raw_emg_imu.csv: 106.68 Hz
  > Processato S08/raw_emg_imu.csv: 191.29 Hz
  > Processato S08/raw_emg_imu.csv: 114.14 Hz
  > Processato S08/raw_emg_imu.csv: 111.47 Hz
  > Processato S08/raw_emg_imu.csv: 109.02 Hz
  > Processato S08/raw_emg_imu.csv: 128.48 Hz
  > Processato S08/raw_emg_imu.csv: 97.37 Hz
  > Processato S08/raw_emg_imu

In [12]:
import pandas as pd
from pathlib import Path
import sys

def processa_imu_log(base_repo_path):
    """
    Esegue i seguenti passaggi:
    1. Cerca 'raw_emg_imu.csv' in tutte le sottocartelle 'S'.
    2. Legge 'Timestamps' e le 6 colonne IMU.
    3. Filtra le righe:
       - Rimuove righe dove TUTTE le 6 colonne IMU sono 0.
       - Rimuove righe con valori IMU duplicati (stessi 6 valori).
    4. Sui dati filtrati, calcola la frequenza:
       - N_valid = (Righe Filtrate Totali) - (Conteggio del primo timestamp)
       - Delta_T = (Ultimo timestamp) - (Primo timestamp) [secondi]
       - Frequenza (Hz) = N_valid / Delta_T
    5. Salva tutte le frequenze in 'frequencies_imu.csv'.
    6. Salva le statistiche in 'frequencies_imu_statistics.txt'.
    7. Salva tutto in ../data/all/myo_inf (relativo allo script).
    """
    
    # 1. Definizione dei percorsi
    
    # Percorso INPUT: Dove si trovano i dati
    base_path = Path(base_repo_path)
    
    # Percorso OUTPUT: Relativo alla posizione dello script
    try:
        script_dir = Path(__file__).resolve().parent
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
    except NameError:
        print("--- ERRORE RILEVATO (Jupyter/Interattivo) ---")
        print("Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.")
        print("Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.")
        script_dir = Path.cwd().resolve()
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
        print("---------------------------------------------")

    # Crea la cartella di output
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Cartella di output creata (o già esistente): {output_dir}")
    except Exception as e:
        print(f"Errore fatale: Impossibile creare la cartella di output {output_dir}.")
        print(f"Dettagli: {e}")
        sys.exit(1)

    # File di output
    output_csv_file = output_dir / "frequencies_imu.csv"
    output_stats_file = output_dir / "frequencies_imu_statistics.txt"

    # Definisci le colonne
    ts_col = 'Timestamps' # Come da tua specifica
    sensor_cols = ['ACC_X', 'ACC_Y', 'ACC_Z', 'GYR_X', 'GYR_Y', 'GYR_Z']
    cols_to_read = [ts_col] + sensor_cols

    # Lista per contenere tutte le frequenze calcolate
    all_frequencies = []
    
    print(f"Inizio ricerca dati in: {base_path}")

    # 2. Trova tutte le cartelle che iniziano per 'S'
    s_folders = [f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')]
    
    if not s_folders:
        print("Attenzione: Nessuna cartella che inizia con 'S' trovata.")
        return

    print(f"Trovate {len(s_folders)} cartelle 'S'. Inizio scansione...")

    # 3. Cerca i file CSV e processali
    file_trovati = 0
    for folder in s_folders:
        for csv_file in folder.rglob("raw_emg_imu.csv"):
            try:
                # Leggi solo le colonne di interesse
                df = pd.read_csv(
                    csv_file, 
                    usecols=cols_to_read, 
                    dtype={ts_col: str}, # Leggi timestamp come stringa
                    skipinitialspace=True
                )
                
                # Rimuovi eventuali righe 'None' o 'NaN'
                df.dropna(inplace=True)
                
                rows_initial = len(df)
                if rows_initial < 2:
                    print(f"  > Attenzione: File ignorato {csv_file} (meno di 2 righe)")
                    continue
                
                # ----------------------------------------------------
                # Inizio logica di FILTRAGGIO
                # ----------------------------------------------------
                
                # 1. Filtro Zeri: Trova righe dove TUTTI i 6 sensori sono 0
                is_all_zero = (df[sensor_cols] == 0).all(axis=1)
                
                # Mantieni solo le righe che NON sono tutte a zero
                df_filtered_zeros = df[~is_all_zero]
                
                rows_after_zeros = len(df_filtered_zeros)
                if rows_after_zeros < 2:
                    print(f"  > Attenzione: File ignorato {csv_file} (nessun dato valido dopo filtro zeri)")
                    continue

                # 2. Filtro Duplicati: Rimuovi duplicati BASATI SUI SENSORI
                df_filtered_final = df_filtered_zeros.drop_duplicates(subset=sensor_cols, keep='first')
                
                rows_final = len(df_filtered_final)
                if rows_final < 2:
                    print(f"  > Attenzione: File ignorato {csv_file} (nessun dato valido dopo filtro duplicati)")
                    continue
                
                # ----------------------------------------------------
                # Inizio logica di CALCOLO (sui dati filtrati)
                # ----------------------------------------------------
                
                # 1. Trova primo e ultimo timestamp (come stringhe)
                first_ts_value = df_filtered_final[ts_col].iloc[0]
                last_ts_value = df_filtered_final[ts_col].iloc[-1]
                
                # 2. Conta le occorrenze del primo timestamp (SUI DATI FILTRATI)
                first_ts_count = (df_filtered_final[ts_col] == first_ts_value).sum()
                
                # 3. Calcola N_valid
                n_valid = rows_final - first_ts_count
                
                if n_valid <= 0:
                    print(f"  > Attenzione: File ignorato {csv_file} (N_valid = {n_valid})")
                    continue

                # 4. Calcola Delta_T (Durata)
                start_time = pd.to_datetime(first_ts_value, format='%H_%M_%S_%f')
                end_time = pd.to_datetime(last_ts_value, format='%H_%M_%S_%f')
                
                if end_time < start_time:
                    end_time += pd.Timedelta(days=1)
                
                delta_t_seconds = (end_time - start_time).total_seconds()
                
                if delta_t_seconds == 0:
                    print(f"  > Attenzione: File ignorato {csv_file} (Durata totale è 0.0s)")
                    continue

                # 5. Calcola Frequenza (Hz)
                frequency_hz = n_valid / delta_t_seconds
                
                all_frequencies.append(frequency_hz)
                file_trovati += 1
                print(f"  > Processato {folder.name}/{csv_file.name}: {frequency_hz:.2f} Hz "
                      f"(Righe: {rows_initial} -> Zeri: {rows_after_zeros} -> Finali: {rows_final})")

            except pd.errors.EmptyDataError:
                print(f"  > Attenzione: File vuoto ignorato {csv_file}")
            except KeyError as e:
                print(f"  > Attenzione: Colonna non trovata in {csv_file}. (Errore: {e})")
            except ValueError as e:
                print(f"  > Errore parsing timestamp in {csv_file}: {e}")
            except Exception as e:
                print(f"  > Errore sconosciuto durante lettura di {csv_file}: {e}")

    print(f"\nScansione completata. Processati {file_trovati} file 'raw_emg_imu.csv'.")

    # 4. Processa e salva i dati
    if all_frequencies:
        print("\n--- Processando frequenze IMU calcolate ---")
        freq_df = pd.DataFrame(all_frequencies, columns=['calculated_imu_frequency_hz'])
        
        # Salva CSV
        try:
            freq_df.to_csv(output_csv_file, index=False)
            print(f"Dati 'frequencies_imu.csv' salvati con successo in:\n{output_csv_file}")
        except Exception as e:
            print(f"Errore duringo il salvataggio di {output_csv_file}: {e}")

        # Calcola e salva statistiche
        try:
            stats_content = f"""
Statistiche per la Frequenza Dati IMU Unici (Hz)
---------------------------------------------------
File 'raw_emg_imu.csv' processati: {file_trovati}

Nota sulla metodologia (come da richiesta):
1. Lette le colonne 'Timestamps' e 6 colonne IMU.
2. Rimosse le righe con tutti i 6 sensori IMU a 0.
3. Rimossi i duplicati basati solo sui 6 valori IMU.
4. N_valid = (Righe Filtrate Totali) - (Conteggio del primo timestamp)
5. Delta_T = (Ultimo timestamp) - (Primo timestamp) [secondi]
6. Frequenza = N_valid / Delta_T
---------------------------------------------------

Metriche aggregate:
  Media:           {freq_df['calculated_imu_frequency_hz'].mean():.3f} Hz
  Deviazione Std:  {freq_df['calculated_imu_frequency_hz'].std():.3f} Hz
  Minimo:          {freq_df['calculated_imu_frequency_hz'].min():.3f} Hz
  25° Percentile:  {freq_df['calculated_imu_frequency_hz'].quantile(0.25):.3f} Hz
  Mediana (50°):   {freq_df['calculated_imu_frequency_hz'].median():.3f} Hz
  75° Percentile:  {freq_df['calculated_imu_frequency_hz'].quantile(0.75):.3f} Hz
  Massimo:         {freq_df['calculated_imu_frequency_hz'].max():.3f} Hz
"""
            with open(output_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            print(f"Statistiche 'frequencies_imu_statistics.txt' salvate con successo in:\n{output_stats_file}")

        except Exception as e:
            print(f"Errore durante il calcolo/salvataggio delle statistiche: {e}")
    else:
        print("\nNessuna frequenza IMU è stata calcolata.")

    print("\n--- Operazione completata ---")

# --- INIZIO SCRIPT ---
if __name__ == "__main__":
    # Percorso INPUT: dove si trovano i dati da analizzare.
    percorso_base_dati = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    processa_imu_log(percorso_base_dati)

--- ERRORE RILEVATO (Jupyter/Interattivo) ---
Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.
Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.
---------------------------------------------
Cartella di output creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf
Inizio ricerca dati in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Trovate 12 cartelle 'S'. Inizio scansione...
  > Processato S03/raw_emg_imu.csv: 46.04 Hz (Righe: 2807 -> Zeri: 2781 -> Finali: 745)
  > Processato S03/raw_emg_imu.csv: 46.89 Hz (Righe: 14052 -> Zeri: 14047 -> Finali: 5407)
  > Processato S03/raw_emg_imu.csv: 47.83 Hz (Righe: 12864 -> Zeri: 12845 -> Finali: 4997)
  > Processato S03/raw_emg_imu.csv: 48.84 Hz (Righe: 25459 -> Zeri: 25456 -> Finali: 13004)
  > Processato S03/raw_emg_imu.csv: 49.19 Hz (Righe: 29451 -> Zeri: 29450 -> Finali: 13585)
  > Processato S08/raw_emg_imu.csv: 49.77 Hz (Righe: 3219 -> Zeri: 3219 -> Final